In [2]:
import ast
import pandas as pd
import glob
import json
import numpy as np
import tqdm
from langchain_text_splitters import MarkdownHeaderTextSplitter
import os

import sys
sys.path.append("../..")
from benchmark.src import create_sentence_nace_code_similarities

In [3]:
all_results_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/similarity_search_descriptions/"

In [4]:
df_overview = pd.read_csv("../../data/datasets/stoxx_600/stoxx_600_overview.csv", sep=";")
df_overview = df_overview.dropna(subset="Report")
df_overview = df_overview.dropna(subset="description_page")

In [5]:
report_jsons = glob.glob("../../data/datasets/stoxx_600/JSONs/*.json")
report_jsons.sort()
report_jsons

['../../data/datasets/stoxx_600/JSONs/AAK AB1.json',
 '../../data/datasets/stoxx_600/JSONs/ABB Ltd.2.json',
 '../../data/datasets/stoxx_600/JSONs/ANDRITZ AG1.json',
 '../../data/datasets/stoxx_600/JSONs/ASM International N.V.1.json',
 '../../data/datasets/stoxx_600/JSONs/ASR Nederland N.V.1.json',
 '../../data/datasets/stoxx_600/JSONs/AXA SA1.json',
 '../../data/datasets/stoxx_600/JSONs/Aalberts N.V.1.json',
 '../../data/datasets/stoxx_600/JSONs/Accelleron Industries AG1.json',
 '../../data/datasets/stoxx_600/JSONs/Acciona SA2.json',
 '../../data/datasets/stoxx_600/JSONs/Accor SA1.json',
 '../../data/datasets/stoxx_600/JSONs/Ackermans & van Haaren NV1.json',
 '../../data/datasets/stoxx_600/JSONs/Adecco Group AG1.json',
 '../../data/datasets/stoxx_600/JSONs/Admiral Group plc1.json',
 '../../data/datasets/stoxx_600/JSONs/Airbus SE1.json',
 '../../data/datasets/stoxx_600/JSONs/Akzo Nobel N.V.2.json',
 '../../data/datasets/stoxx_600/JSONs/Alcon AG1.json',
 '../../data/datasets/stoxx_600/JS

In [6]:
queries = [
    "The Companies Business Model",
    "Our business areas",
    "Fundamental Information about the Group",
    "Overview over the Group",
    "What we do",
    "We produce",
]


In [7]:
negatives = [
    "Table of contents",
    "Financial performance",
]

In [7]:
def process_markdown_with_headers(markdown_text, headers_to_split_on):
    """
    Splits a markdown document into chunks based on specified headers and includes header metadata in the content.

    Args:
        markdown_text (str): The markdown text to process.
        headers_to_split_on (list): A list of tuples specifying the headers to split on.

    Returns:
        list: A list of processed chunks with header metadata included in the content.
    """
    from langchain_text_splitters import MarkdownHeaderTextSplitter

    # Initialize the splitter
    splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

    # Split into chunks
    chunks = splitter.split_text(markdown_text)

    # Include the headers in the text content
    final_chunks = []
    for chunk in chunks:
        # Join all headers from metadata into one string
        header_str = " ".join(
            f"{key}: {value}" for key, value in chunk.metadata.items() if value
        )
        # Prepend header info to text
        full_text = header_str + "\n\n" + chunk.page_content
        final_chunks.append(full_text)

    return final_chunks

def process_markdown_pages(markdown_pages, headers_to_split_on):
    """
    Processes a list of markdown pages, splits each into chunks, and returns a DataFrame.
    Args:
        markdown_pages (list): A list of markdown strings representing pages.
        headers_to_split_on (list): A list of tuples specifying the headers to split on.
    Returns:
        pd.DataFrame: A DataFrame with columns 'chunk', 'page', and 'id'.
    """
    all_chunks = []
    for page_index, markdown_text in enumerate(markdown_pages):
        chunks = process_markdown_with_headers(markdown_text, headers_to_split_on)
        for chunk_index, chunk in enumerate(chunks):
            all_chunks.append({
                'chunk': chunk,
                'page': page_index,
                'id': chunk_index
            })
    return pd.DataFrame(all_chunks)


In [31]:
def create_similarities(queries: list, texts: list) -> np.array: 

    embedder = create_sentence_nace_code_similarities.get_embedder()
    
    report_embeddings = create_sentence_nace_code_similarities.embed_document(embedder, texts)

    query_embeddings = create_sentence_nace_code_similarities.embed_document(embedder, queries)

    res_similarities = []

    for query_embedding in query_embeddings:
        cos_sim_to_query = lambda x: create_sentence_nace_code_similarities.cosine_similarity(x, query_embedding)
        similarities = np.apply_along_axis(cos_sim_to_query, axis=1, arr = report_embeddings)
        res_similarities.append(similarities)

    #df_report_similarities = pd.DataFrame(data=np.array([similarities, texts, np.arange(len(texts))]).T, columns=["Similarities", "Text", "Page_nr"])

    return np.array(res_similarities)

In [9]:
def create_similarities_titles(queries: list, texts: list) -> np.array: 

    embedder = create_sentence_nace_code_similarities.get_embedder()

    get_titles = lambda x: [text for text in x.split("\n") if len(text) > 0 and text[0] == "#"]
    titles = ["".join(get_titles(text)) for text in texts]
    titles = [title if title != "" else "false" for title in titles]
    
    report_embeddings = create_sentence_nace_code_similarities.embed_document(embedder, titles)

    query_embeddings = create_sentence_nace_code_similarities.embed_document(embedder, queries)

    res_similarities = []

    for query_embedding in query_embeddings:
        cos_sim_to_query = lambda x: create_sentence_nace_code_similarities.cosine_similarity(x, query_embedding)
        similarities = np.apply_along_axis(cos_sim_to_query, axis=1, arr = report_embeddings)
        res_similarities.append(similarities)

    return np.array(res_similarities)

In [10]:
def create_structural_score(texts: list) -> np.array: 

    # minimum words
    min_words = lambda x: int(len(x.split(" ")) > 50) * 10

    return list(map(min_words, texts))

In [11]:
def print_similarities(df_report_similarities): 
    df_report_similarities = df_report_similarities.sort_values("score", ascending=False)
    text = "" 
    for i, row in df_report_similarities.iterrows(): 
        text += f'Sim {row["score"]}:\n\n{row["text"]}\n' + 200 * "-" + '\n\n'
    print(text)
    return text

In [12]:
def filter_report(texts, queries, negatives):

    queries_sim = create_similarities(queries+negatives, texts)
    queries_sim *= np.concatenate([np.ones((len(texts), len(queries))),-np.ones((len(texts),len(negatives)))], axis=1).T
    struct_score = create_structural_score(texts)

    df = pd.DataFrame(np.concatenate([queries_sim, np.array([struct_score])]).T, columns=queries+negatives+["struct_score"])

    df["score"] = df.sum(1)
    df["text"] = texts
    df["page"] = np.arange(len(texts))
    
    df = df.sort_values("score", ascending=False)

    return df

In [13]:
def filter_report_title_and_text(texts, queries, negatives, titles=True):

    queries_sim = create_similarities(queries+negatives, texts)
    queries_sim *= np.concatenate([np.ones((len(texts), len(queries))),-np.ones((len(texts),len(negatives)))], axis=1).T
    struct_score = create_structural_score(texts)

    if titles: 
        queries_titles_sim = create_similarities_titles(queries+negatives, texts)
        queries_titles_sim *= np.concatenate([np.ones((len(texts), len(queries))),-np.ones((len(texts),len(negatives)))], axis=1).T
        titles_columns = [q + "_title" for q in queries] + [n + "_title" for n in negatives]    
        df = pd.DataFrame(np.concatenate([queries_sim, queries_titles_sim, np.array([struct_score])]).T, columns=queries+negatives+titles_columns+["struct_score"])
    else: 
        df = pd.DataFrame(np.concatenate([queries_sim, np.array([struct_score])]).T, columns=queries+negatives+["struct_score"])

    df["score"] = df.sum(1)
    df["text"] = texts
    df["page"] = np.arange(len(texts))
    
    df = df.sort_values("score", ascending=False)

    return df

In [14]:
def filter_report_title_and_text_on_chunks(texts, queries, negatives, titles=True):

    df_chunks = process_markdown_pages(texts, [("#", "#"), ("##", "##")])

    queries_sim = create_similarities(queries+negatives, df_chunks["chunk"])
    queries_sim *= np.concatenate([np.ones((len(df_chunks["chunk"]), len(queries))),-np.ones((len(df_chunks["chunk"]),len(negatives)))], axis=1).T
    struct_score = create_structural_score(df_chunks["chunk"])

    if titles: 
        queries_titles_sim = create_similarities_titles(queries+negatives, df_chunks["chunk"])
        queries_titles_sim *= np.concatenate([np.ones((len(df_chunks["chunk"]), len(queries))),-np.ones((len(df_chunks["chunk"]),len(negatives)))], axis=1).T
        titles_columns = [q + "_title" for q in queries] + [n + "_title" for n in negatives]    
        df = pd.DataFrame(np.concatenate([queries_sim, queries_titles_sim, np.array([struct_score])]).T, columns=queries+negatives+titles_columns+["struct_score"])
    else: 
        df = pd.DataFrame(np.concatenate([queries_sim, np.array([struct_score])]).T, columns=queries+negatives+["struct_score"])

    df["score"] = df.sum(1)
    df["text"] = df_chunks["chunk"]
    df["page"] = df_chunks["page"]
    df["id"] = df_chunks["id"]
    
    df = df.sort_values("score", ascending=False)

    return df

In [15]:
def filter_report_titles(texts, queries, negatives):

    get_titles = lambda x: [text for text in x.split("\n") if len(text) > 0 and text[0] == "#"]
    titles = ["".join(get_titles(text)) for text in texts]
    titles = [title if title != "" else "false" for title in titles]

    queries_sim = create_similarities_titles(queries+negatives, titles)
    queries_sim *= np.concatenate([np.ones((len(titles), len(queries))),-np.ones((len(titles),len(negatives)))], axis=1).T
    #struct_score = create_structural_score(texts)
    
    #df = pd.DataFrame(np.concatenate([queries_sim, np.array([struct_score])]).T, columns=queries+ negatives+["struct_score"])
    df = pd.DataFrame(queries_sim.T, columns=queries+negatives)
    df["score"] = df.sum(1)
    df["titles"] = titles
    df["text"] = texts
    df["page"] = np.arange(len(texts))
    
    df = df.sort_values("score", ascending=False)

    return df

In [ ]:
def filter_report_title_and_text_with_nace_classe(texts, queries, negatives, nace_class_description, titles=True):

    queries_sim = create_similarities([nace_class_description]+queries+negatives, texts)
    queries_sim *= np.concatenate([np.ones((len(texts), 1+len(queries))),-np.ones((len(texts),len(negatives)))], axis=1).T
    struct_score = create_structural_score(texts)

    if titles:
        queries_titles_sim = create_similarities_titles(queries+negatives, texts)
        queries_titles_sim *= np.concatenate([np.ones((len(texts), len(queries))),-np.ones((len(texts),len(negatives)))], axis=1).T
        titles_columns = [q + "_title" for q in queries] + [n + "_title" for n in negatives]    
        df = pd.DataFrame(np.concatenate([queries_sim, queries_titles_sim, np.array([struct_score])]).T, columns=queries+negatives+titles_columns+["struct_score"])
    else: 
        df = pd.DataFrame(np.concatenate([queries_sim, np.array([struct_score])]).T, columns=queries+negatives+["struct_score"])

    df["score"] = np.sum([df[nace_class_description], df[queries].mean(0), df[negatives].mean(0), df["struct_score"]], axis=1)
    df["text"] = texts
    df["page"] = np.arange(len(texts))
    
    df = df.sort_values("score", ascending=False)

    return df

Test the results of above

In [ ]:
def generate_results(report_jsons, df_overview, filter_function, store_at, until_page = 50, nace_class_description):
    
    results = dict()
    results_overview = []
    # generate dfs with scores

    os.makedirs(store_at, exist_ok=True)

    for i, row in tqdm.tqdm(df_overview[27:].iterrows()): 

        try: 
            report_json = list(filter(lambda x: row["Report"].replace(".pdf", "") in x, report_jsons))[0]
        except IndexError: 
            continue

        with open(report_json, "r") as f:
            report =  json.load(f)
    
        texts = [page["markdown"] for page in report["pages"]][:until_page]
        texts = [t if t != "" else "false" for t in texts]

        df = filter_function(texts, queries, negatives)

        results[os.path.basename(report_json)] = df 

        df.to_csv(store_at + "/" + os.path.basename(report_json).replace(".json", ".csv"))

        if len(row) > 0 and row["description_page"] is not np.nan:
            description_page = ast.literal_eval(row["description_page"])
            description_page = description_page if isinstance(description_page, list) else [description_page]
            try:
                page_and_position = dict()
                for page in description_page:
                    page_and_position[page] = df["page"].tolist().index(page)
                results_overview.append({"Report": row["Report"], "Best Page Position": min(page_and_position.values()), "all_positions": page_and_position})

            except ValueError as e: 
                print(e)
                results_overview.append({"Report": row["Report"]})
        else: 
            results_overview.append({"Report": row["Report"]})

        pd.DataFrame(results_overview, index=np.arange(len(results_overview))).to_csv(store_at + "/" + "results_overview.csv")

    return results

In [17]:
generate_results(report_jsons, df_overview, filter_report_title_and_text, store_at=os.path.join(all_results_path, "title_and_text"))

0it [00:00, ?it/s]Token indices sequence length is longer than the specified maximum sequence length for this model (763 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (767 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (706 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (784 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (865 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum seque

{'Aviva plc1.json':     The Companies Business Model  Our business areas  \
 35                      0.488983            0.421156   
 24                      0.481905            0.406629   
 29                      0.521203            0.442223   
 26                      0.414898            0.421682   
 28                      0.530735            0.400130   
 25                      0.499570            0.400676   
 44                      0.458158            0.461019   
 40                      0.486715            0.437677   
 13                      0.353749            0.327138   
 8                       0.377210            0.357099   
 31                      0.424618            0.429646   
 4                       0.467595            0.347741   
 41                      0.431194            0.441961   
 32                      0.415365            0.350873   
 5                       0.457036            0.363316   
 6                       0.515507            0.425206   
 30         

In [18]:
#generate_results(report_jsons, df_overview, filter_report_title_and_text, store_at=os.path.join(all_results_path, "title_and_text__all_reports_2"))

In [19]:
#generate_results(report_jsons, df_overview, filter_report_titles, store_at=os.path.join(all_results_path, "title__all_reports"))

In [20]:
#generate_results(report_jsons, df_overview, filter_report, store_at=os.path.join(all_results_path, "text__all_reports"))

In [21]:
#generate_results(report_jsons, df_overview, filter_report_title_and_text_on_chunks, store_at=os.path.join(all_results_path, "text_and_title_on_chunks"))

In [22]:

#df_chunks = process_markdown_pages(texts, [("#", "#"), ("##", "##")])

In [34]:
with open(report_jsons[0], "r") as f: 
    texts_json = json.load(f)
texts = [text["markdown"] for text in texts_json["pages"]]

In [23]:
nace_letter = df_overview[df_overview["Report"] == texts["source_file"]]["NACE_letter"].iloc[0]

In [27]:
nace_descriptions = pd.read_csv("../../data/NACE_Rev2_Structure_Explanatory_Notes_EN__1_.tsv", sep="\t")
nace_class_description = nace_descriptions[nace_descriptions["CODE"] == nace_letter]["Includes"].iloc[0]

In [40]:
filter_report_title_and_text_with_nace_classe(texts, queries, negatives, nace_class_description, titles=True)

Token indices sequence length is longer than the specified maximum sequence length for this model (597 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (881 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (720 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (604 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (738 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for thi

NameError: name 'create_structural_score' is not defined

In [38]:
texts = [text if text!="" else "false" for text in texts]

In [39]:
nace_sim = create_similarities(nace_class_description, texts)

Token indices sequence length is longer than the specified maximum sequence length for this model (597 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (881 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (720 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (604 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (738 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for thi

KeyboardInterrupt: 